# PolOCRBench: Kraken na Kaggle GPU

Włącz **Internet** oraz **GPU T4** w ustawieniach notebooka, następnie Run All. Notebook wykonuje inferencję, nie trening: preflight, test jednej strony, 36 stron, walidacja, metryki i ZIP dowodowy.

Model: recognizer i segmenter fine-tunowane na EHRI. To inna konfiguracja niż historyczny baseline z domyślnym segmenterem BLLA. Wynik jakości nie jest jeszcze znany. Referencje mają znane problemy Unicode; nie są tutaj poprawiane.

Opcjonalny sekret Kaggle `HF_TOKEN` służy wyłącznie do odczytu publicznych repozytoriów. Notebook niczego nie publikuje. Każde Run All tworzy nowy katalog.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, subprocess, sys, json, hashlib, shutil, zipfile

CODE_REVISION = "8c5cfd1a9be1f2eac6284ee77500dbdd748055ad"
DATA_REVISION = "a2480fde6f15284701458ff370b81cce50dc5c2d"
MODEL_REVISION = "1947d4c107936ed59ded16446a77b6587fa3874a"
WORK = Path("/kaggle/working") / ("polocrbench-kraken-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
WORK.mkdir(parents=True, exist_ok=False)
REPO = WORK / "OCR_engine"
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

subprocess.run(["git", "clone", "https://github.com/PiotrStyla/OCR_engine.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_REVISION], check=True)
assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == CODE_REVISION
print("Pinned code:", CODE_REVISION)
print("Output:", WORK)

## Środowisko i rzeczywisty test CUDA

Wersje wszystkich zainstalowanych pakietów zostaną zapisane w artefaktach. Jeżeli instalacja wymaga restartu kernela, zrestartuj go i uruchom notebook ponownie od początku.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "kraken==7.1.1", "jiwer==4.0.0", "huggingface_hub"], check=True)
import importlib.metadata
import torch
assert importlib.metadata.version("kraken") == "7.1.1"
if not torch.cuda.is_available():
    raise RuntimeError("Włącz GPU T4 w ustawieniach Kaggle.")
# Exercises the CUDA kernels, not just device discovery.
probe = torch.ones((16, 16), device="cuda")
assert (probe @ probe).sum().item() == 4096
torch.cuda.synchronize()
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
(WORK / "pip-freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")

def run_module(module, *arguments, quiet=False):
    result = subprocess.run([sys.executable, "-m", module, *map(str, arguments)],
                            cwd=REPO, check=True, text=True,
                            stdout=subprocess.PIPE if quiet else None)
    return result.stdout

## Przypięte dane i modele

Pobierane są tylko paczka testowa oraz dwa pliki wag. Weryfikacja SHA-256 poprzedza inferencję. Obrazy są konwertowane do PNG bez zmiany zdekodowanych pikseli.

In [ ]:
from huggingface_hub import hf_hub_download
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        token = None

bundle = hf_hub_download("PiotrSty/impact-print-v2", "impact-print-v2-test.tar.gz",
                         repo_type="dataset", revision=DATA_REVISION, token=token)
recognizer = hf_hub_download("PiotrSty/ehri-dataset", "models/polish_nfd_finetuned.safetensors",
                             repo_type="dataset", revision=MODEL_REVISION, token=token)
segmenter = hf_hub_download("PiotrSty/ehri-dataset", "models/polish_seg_best.safetensors",
                            repo_type="dataset", revision=MODEL_REVISION, token=token)
RECOGNIZER_HASH = "9b731f0694af3c0c4c6f31a689fc4f2788292bab31eedaad85184698ee8f3ff6"
SEGMENTER_HASH = "a817f2a1d3a0eee51b456008ba9952afe60b2ade712f2e7d75ec1608d008ea86"
for filename, expected in [(recognizer, RECOGNIZER_HASH), (segmenter, SEGMENTER_HASH)]:
    assert hashlib.sha256(Path(filename).read_bytes()).hexdigest() == expected, "Weight checksum mismatch"

ORIGINAL = WORK / "original"
PNG = WORK / "png"
run_module("training.stage_impact_benchmark", "--archive", bundle, "--output", ORIGINAL)
run_module("training.prepare_ocr_images", "--manifest", ORIGINAL / "manifest.jsonl", "--output", PNG)
MANIFEST = PNG / "manifest.jsonl"
run_module("training.run_kraken_benchmark", "--manifest", MANIFEST, "--output", WORK / "preflight")
model_args = ["--recognizer", recognizer, "--recognizer-sha256", RECOGNIZER_HASH,
              "--segmenter", segmenter, "--segmenter-sha256", SEGMENTER_HASH]
print("Data and weights verified.")

## Smoke-test jednej strony

Pełny pomiar ruszy tylko po udanym odczycie strony testowej. `status=ok` potwierdza wykonanie inferencji, nie poprawność transkrypcji.

In [ ]:
rows = [json.loads(line) for line in MANIFEST.read_text(encoding="utf-8").split("\n") if line.strip()]
assert len(rows) == 36
SMOKE_MANIFEST = PNG / "smoke-manifest.jsonl"
SMOKE_MANIFEST.write_text(json.dumps(rows[0], ensure_ascii=False) + "\n", encoding="utf-8")
SMOKE = WORK / "smoke"
run_module("training.run_kraken_benchmark", "--manifest", SMOKE_MANIFEST, "--output", SMOKE,
           "--execute", *model_args)
smoke_run = json.loads((SMOKE / "run.json").read_text())
smoke_prediction = json.loads((SMOKE / "predictions.jsonl").read_text(encoding="utf-8"))
assert smoke_run["state"] == "completed" and smoke_run["pages_completed"] == 1
if smoke_prediction["status"] != "ok":
    raise RuntimeError("Smoke failed: " + smoke_prediction.get("error_type", "unknown"))
print("Smoke passed. Characters returned:", len(smoke_prediction["text"]))

## Pełne 36 stron i ocena

Wyniki błędne i puste pozostają w mianowniku. Protokół historyczny i nowy są liczone osobno. Ocena struktury Markdown jest diagnostyczna; ten zbiór nie ma niezależnych anotacji struktury Markdown.

In [ ]:
FULL = WORK / "full"
run_module("training.run_kraken_benchmark", "--manifest", MANIFEST, "--output", FULL,
           "--execute", *model_args)
run_module("training.validate_submission", "--manifest", MANIFEST,
           "--predictions", FULL / "predictions.jsonl", "--output", FULL / "validation.json")
run_module("training.transcription_eval", "--manifest", MANIFEST,
           "--predictions", FULL / "predictions.jsonl", "--output", FULL / "transcription-v1.1.json")
run_module("training.benchmark_pages", "--manifest", MANIFEST,
           "--predictions", FULL / "predictions.jsonl", "--output", FULL / "legacy-metrics.json", quiet=True)
metadata = json.loads((FULL / "run.json").read_text())
assert metadata["state"] == "completed" and metadata["pages_completed"] == 36
for name in ("legacy-metrics.json", "transcription-v1.1.json"):
    report = json.loads((FULL / name).read_text(encoding="utf-8"))
    print(name, {key: report[key] for key in ("pages", "errors_or_missing", "cer_micro", "wer_micro")})
print("Page execution errors:", metadata["error_pages"])

## Pobranie dowodów

ZIP zawiera predykcje, metryki, wersje środowiska i manifesty pochodzenia. Nie zawiera wag ani skanów; można je odtworzyć z przypiętych źródeł. Pobierz ZIP z panelu Output przed zakończeniem sesji.

In [ ]:
EVIDENCE = WORK / "evidence"
EVIDENCE.mkdir()
for folder in (SMOKE, FULL, WORK / "preflight"):
    shutil.copytree(folder, EVIDENCE / folder.name)
shutil.copyfile(MANIFEST, EVIDENCE / "input-manifest.jsonl")
shutil.copyfile(PNG / "conversion.json", EVIDENCE / "conversion.json")
shutil.copyfile(ORIGINAL / "verification.json", EVIDENCE / "verification.json")
shutil.copyfile(WORK / "pip-freeze.txt", EVIDENCE / "pip-freeze.txt")
provenance = {"code_revision": CODE_REVISION, "data_revision": DATA_REVISION,
              "model_revision": MODEL_REVISION, "recognizer_sha256": RECOGNIZER_HASH,
              "segmenter_sha256": SEGMENTER_HASH,
              "scope": "EHRI finetuned segmenter + recognizer; historical print diagnostic, not SOTA",
              "license": "Reference text: IMPACT/PSNC, CC BY 3.0",
              "image_reproduction": "Rebuild portable PNG manifest using the pinned code and archive."}
(EVIDENCE / "provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
checksums = {path.relative_to(EVIDENCE).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
             for path in sorted(EVIDENCE.rglob("*")) if path.is_file()}
(EVIDENCE / "checksums.json").write_text(json.dumps(checksums, indent=2), encoding="utf-8")
zip_path = WORK / "polocrbench-kraken-evidence.zip"
with zipfile.ZipFile(zip_path, "x", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(EVIDENCE.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(EVIDENCE).as_posix())
print("Download:", zip_path)
from IPython.display import FileLink, display
display(FileLink(str(zip_path)))